# embodied-efficiency — VLA flow-matching kernel on a free T4

**Runtime → Change runtime type → T4 GPU** (free), then *Run all*.

Produces: fp16 baseline (eager / CUDA-graphs via torch.compile / manual CUDA graph), the INT8/INT4 fused-kernel latency, CUDA-graph + low-bit composition, and correctness + no-leak evals. Free/unbilled — standard **T4** only, never a premium accelerator.

In [ ]:
import torch
print('torch', torch.__version__)
print('gpu  ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set runtime to T4')
try:
    import triton; print('triton', triton.__version__)
except ImportError:
    !pip -q install triton

In [ ]:
# Private repo: clone with a fine-grained read-only token, e.g.
#   REPO = 'https://<YOUR_PAT>@github.com/LaelaZorana/embodied-efficiency.git'
# (or upload the kernel/ folder, or flip the repo public once validated).
REPO = 'https://github.com/LaelaZorana/embodied-efficiency.git'
!git clone -q $REPO
%cd embodied-efficiency

## 1. fp16 baseline — eager vs torch.compile(CUDA-graphs) vs manual CUDA graph

In [ ]:
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --compile --graph --peak T4

## 2. Low-bit fused kernels, and CUDA graph + low-bit composed (the orthogonal stack)

In [ ]:
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --quant int8 --graph --peak T4
!python kernel/bench.py --steps 10 4 8 --batch 1 --dtype fp16 --quant int4 --graph --peak T4

## 3. Correctness + no-leak evals
Kernel numerics vs torch reference; CUDA-graph correctness, stale-input, and memory-leak checks; quant fidelity.

In [ ]:
!python kernel/triton_gemm.py   # triton vs torch-dequant; action rMSE
!python kernel/cudagraph.py     # graph correctness + stale-input + no-leak
!python kernel/quant.py         # weight-byte reduction + fidelity table

## What to read off
- §1 `eager` vs `compile_reduce_overhead` vs `graph` ms/step → how much is launch overhead (graphs remove it).
- §2 `int8/graph`, `int4/graph` ms/step → the stacked win; compare to the **1.97× / 3.88×** weight-traffic ceilings.
- §3 must print `Triton kernel correctness ✓` and `CUDA-graph correctness + stale-input + no-leak ✓`, with action rMSE 0.0025 (int8) / 0.0423 (int4).
- If any eval prints `FAIL`, the kernel/graph is not trustworthy yet — fix before quoting latency.